# Regex practice: predict, test, revise

Run from `notebooks/`. During the whiteboard activity, solve your assigned problem on paper **before** running code. Then use the cells to test your pattern. Later, work with a partner on the citation challenge. These four problems check *whole fields*.

In [ ]:
import re
from pathlib import Path
import pandas as pd

## Group A: ZIP codes

Write a pattern for five digits with an optional four-digit extension. Explain why the rejected value fails.

In [ ]:
zip_pattern = r""  # TODO: write your group A pattern
zip_cases = {
    "84043": True,
    "84043-4820": True,
    "84043-482": False,
}
for value, expected in zip_cases.items():
    observed = bool(re.fullmatch(zip_pattern, value))
    print(f"{value:12} expected={expected!s:5} observed={observed}")

## Group B: phone numbers

Allow a hyphen or dot as each separator. Does your rule also allow mixed separators?

In [ ]:
phone_pattern = r""  # TODO: write your group B pattern
phone_cases = {
    "801-555-0142": True,
    "801.555.0142": True,
    "8015550142": False,
}
for value, expected in phone_cases.items():
    observed = bool(re.fullmatch(phone_pattern, value))
    print(f"{value:17} expected={expected!s:5} observed={observed}")

## Group C: a narrow campus email

Allow letters and numbers in the username and require the literal domain `byu.edu`. A real email pattern would need more tests.

In [ ]:
email_pattern = r""  # TODO: write your group C pattern
email_cases = {"tom@byu.edu": True, "tom7@byu.edu": True, "tom@byuXedu": False}
for value, expected in email_cases.items():
    print(f"{value:16} expected={expected!s:5} observed={bool(re.fullmatch(email_pattern, value))}")

## Group D: a two-book citation

Allow only John and Alma with an ordinary space, a chapter, a colon, and a verse. We will stress the rule with actual text next.

In [ ]:
simple_citation = r""  # TODO: write your group D pattern
citation_cases = {"John 8:31": True, "Alma 34:32": True, "Mosiah 3:19": False}
for value, expected in citation_cases.items():
    print(f"{value:14} expected={expected!s:5} observed={bool(re.fullmatch(simple_citation, value))}")

## Think, pair, share: an assistant's citation pattern

Predict the matches. Find a missed reference and a misleading partial match. With a partner, revise the specification before changing the code. Each pair should propose one new test case that could expose another failure.

In [ ]:
suggested = r"[A-Za-z]+ \d+:\d+"
cases = [
    "John 8:31–32",
    "1\u00a0Nephi 8:27",
    "D&C 19:16",
    "Doctrine and Covenants 121:35",
    "Alma 7:12",
]
for value in cases:
    print(repr(value), "->", re.findall(suggested, value))

## Try your improved rule on actual talks

The two records include numbered books, verse ranges with an en dash, and both long and abbreviated forms of Doctrine and Covenants. Begin with the working `John` pattern and extend it. Use named groups for `book`, `chapter`, `verse_start`, and optional `verse_end`.

In [ ]:
talks = pd.read_csv(Path("../data/talks_sample.csv"))
print(talks[["talk_name", "conference"]].to_string(index=False))
pattern = r"(?P<book>John)\s+(?P<chapter>\d+):(?P<verse_start>\d+)"  # TODO
found = talks["text"].str.extractall(pattern)
print(found.head(10).to_string())
print("Matches by talk:", found.groupby(level=0).size().to_dict())

## Evaluate and normalize

After revising the pattern, inspect a few original spans. Decide which values require normalization in a *new* column, keeping the raw text. The completed walkthrough demonstrates one approach if you become stuck.

In [ ]:
# Explore your new results here. For example:
for talk_row, row in talks.iterrows():
    spans = list(re.finditer(pattern, row["text"]))
    print(row["talk_name"], "matches:", len(spans))
    print([m.group() for m in spans[:4]])

### Reflection

What does your pattern still miss? What would be a false match? Which part of the work belongs in a subsequent standardization step rather than inside the regex?